# 08 — Offline Evaluation & Model Comparison

**Purpose:** Unified offline evaluation of all trained recommender models.
Compares performance against the heuristic baseline using the standard
BDC metric suite.

## Evaluation Protocol

- **Temporal split:** Last 20% of interactions per user form the test set
- **Metrics:** Precision@5, Recall@5, nDCG@5, ILD, Novelty
- **Ground truth:** `gold_user_item_matrix` (node IDs the user interacted with in test period)

## Models Evaluated

| Model | Slate file |
|-------|------------|
| Heuristic Baseline | N/A (placeholder from guide) |
| Cosine CF | `output/slates/cf_cosine_slates.csv` |
| Implicit ALS | `output/slates/als_slates.csv` |
| SRD-GRU4Rec | `output/slates/srd_slates.csv` |
| DivKG-RL | `output/slates/rl_slates.csv` |

## Expected Baseline (from DATA_ANALYST_RECOMMENDER_GUIDE.md)

| Metric | Heuristic Baseline |
|--------|-------------------|
| Precision@5 | 0.40 |
| Recall@5 | 0.25 |
| nDCG@5 | 0.38 |
| ILD | 0.72 |
| Novelty | 1.20 |

In [ ]:
import os
import sys
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR    = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
SLATES_DIR  = '../output/slates'
EVAL_DIR    = '../output/evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)

%matplotlib inline
print('Environment ready.')

## 1. Load Ground Truth (Temporal Split)

In [ ]:
from scripts.load_data import train_test_split_temporal
from scripts.metrics import evaluate_model, summarize_evaluation, build_popularity_distribution

def load_user_item_data(gold_dir: str) -> pd.DataFrame:
    path = os.path.join(gold_dir, 'gold_user_item_matrix.parquet')
    if os.path.exists(path):
        print(f'[Parquet] Loading from {path}')
        return pd.read_parquet(path)
    print('[Demo] Generating synthetic data...')
    rng = np.random.default_rng(42)
    n = 3000
    return pd.DataFrame({
        'user_id':    rng.integers(1, 101, n),
        'node_id':    rng.integers(100, 160, n),
        'implicit_affinity_score': rng.uniform(0.1, 5.0, n).round(3),
        'created_at': pd.date_range('2024-01-01', periods=n, freq='20min'),
    })

df_all = load_user_item_data(GOLD_DIR)

if 'created_at' not in df_all.columns:
    df_all['created_at'] = pd.date_range('2024-01-01', periods=len(df_all), freq='10min')

# Temporal split: last 20% per user -> test set
train_df, test_df = train_test_split_temporal(df_all, test_frac=0.2)

ground_truth = (
    test_df.groupby('user_id')['node_id']
    .apply(list)
    .to_dict()
)

# Item popularity for novelty metric
item_popularity = build_popularity_distribution(train_df)

print(f'Train interactions: {len(train_df):,}')
print(f'Test interactions:  {len(test_df):,}')
print(f'Test users:         {len(ground_truth):,}')
print(f'Unique items:       {df_all["node_id"].nunique():,}')

## 2. Load Model Predictions from Slate CSVs

In [ ]:
SLATE_FILES = {
    'Cosine CF':    os.path.join(SLATES_DIR, 'cf_cosine_slates.csv'),
    'Implicit ALS': os.path.join(SLATES_DIR, 'als_slates.csv'),
    'SRD-GRU4Rec':  os.path.join(SLATES_DIR, 'srd_slates.csv'),
    'DivKG-RL':     os.path.join(SLATES_DIR, 'rl_slates.csv'),
}

model_predictions = {}

for model_name, path in SLATE_FILES.items():
    if os.path.exists(path):
        df_slates = pd.read_csv(path)
        if 'recommended_node_ids' in df_slates.columns:
            preds = {}
            for _, row in df_slates.iterrows():
                try:
                    node_list = ast.literal_eval(str(row['recommended_node_ids']))
                    preds[row['user_id']] = [int(x) for x in node_list]
                except Exception:
                    preds[row['user_id']] = []
            model_predictions[model_name] = preds
            print(f'[Loaded] {model_name}: {len(preds):,} users from {path}')
        else:
            print(f'[Skip]   {model_name}: unexpected CSV schema in {path}')
    else:
        print(f'[Missing] {model_name}: {path}')
        print(f'          Run the corresponding training notebook first.')

print(f'\nModels with slates loaded: {list(model_predictions.keys())}')

## 3. Evaluate Each Model

In [ ]:
K = 5
comparison_rows = []

# Heuristic baseline row (from DATA_ANALYST_RECOMMENDER_GUIDE.md)
comparison_rows.append({
    'model':        'Heuristic Baseline',
    'precision@5':  0.40,
    'recall@5':     0.25,
    'ndcg@5':       0.38,
    'ild':          0.72,
    'novelty':      1.20,
    'source':       'guide_placeholder',
})

# Evaluate loaded model predictions
for model_name, predictions in model_predictions.items():
    # Filter to users with ground truth
    filtered_preds = {u: v for u, v in predictions.items() if u in ground_truth}

    if not filtered_preds:
        print(f'[Skip] {model_name}: no predictions for test users')
        continue

    eval_df = evaluate_model(
        filtered_preds,
        ground_truth,
        k=K,
        item_popularity=item_popularity,
    )
    summary = summarize_evaluation(eval_df)

    row = {'model': model_name, 'source': 'computed'}
    for metric in ['precision@5', 'recall@5', 'ndcg@5', 'ild', 'novelty']:
        matching_cols = [c for c in eval_df.columns
                         if metric.replace('@5', '').lower() in c.lower()]
        if matching_cols:
            row[metric] = eval_df[matching_cols[0]].mean()
        else:
            row[metric] = float('nan')

    comparison_rows.append(row)
    print(f'[Evaluated] {model_name}: precision@5={row["precision@5"]:.4f}, '
          f'recall@5={row["recall@5"]:.4f}, ndcg@5={row["ndcg@5"]:.4f}')

comparison_df = pd.DataFrame(comparison_rows).set_index('model')
metric_cols = ['precision@5', 'recall@5', 'ndcg@5', 'ild', 'novelty']

print('\n=== Model Comparison Table ===')
display(comparison_df[metric_cols].round(4))

## 4. Grouped Bar Chart

In [ ]:
# Filter only numeric metric columns without NaN rows
plot_df = comparison_df[metric_cols].copy()
display_metrics = [m for m in metric_cols if not plot_df[m].isna().all()]

n_models  = len(plot_df)
n_metrics = len(display_metrics)
x = np.arange(n_metrics)
bar_width = 0.75 / n_models

colors = plt.cm.tab10(np.linspace(0, 0.9, n_models))

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model_name, row) in enumerate(plot_df.iterrows()):
    values = [row[m] if not pd.isna(row[m]) else 0.0 for m in display_metrics]
    offset = (i - n_models / 2) * bar_width + bar_width / 2
    bars = ax.bar(x + offset, values, bar_width * 0.9,
                  label=model_name, color=colors[i], edgecolor='white', alpha=0.9)

ax.set_title('Offline Model Comparison — Recommender Metrics (@K=5)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in display_metrics], fontsize=11)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, max(plot_df[display_metrics].max()) * 1.25)
plt.tight_layout()
plt.savefig('../output/model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to ../output/model_comparison_chart.png')

## 5. Save Comparison Results

In [ ]:
out_path = os.path.join(EVAL_DIR, 'model_comparison.csv')
comparison_df[metric_cols].reset_index().to_csv(out_path, index=False)
print(f'Model comparison table saved to: {out_path}')

# Final formatted display
print()
print('=== Final Model Comparison ===')
print(comparison_df[metric_cols].round(4).to_string())

# Highlight best model per metric
print()
print('=== Best Model per Metric ===')
for m in metric_cols:
    if not comparison_df[m].isna().all():
        best_model = comparison_df[m].idxmax()
        best_val   = comparison_df[m].max()
        print(f'  {m:<15}: {best_model} ({best_val:.4f})')